# Notebook 06: Hensen Open Data Audit

This notebook performs a detailed audit of the Hensen et al. (2015) Delft loophole-free Bell-test dataset. 
It verifies the loading process, filtering steps, and reproduction of the published CHSH result.

> **Scientific Scope:** This notebook validates the open-data loading, filtering, setting-pair mapping, and CHSH reconstruction pipeline. It does not claim that the Hensen experiment measured X-Theta spacetime holonomy, because the dataset does not contain the required spacetime-path or gravitational metadata.

In [ ]:
from __future__ import annotations
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# Standardized project root addition
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from xtheta.data.adapters.hensen import load_hensen_dataset
from xtheta.data.validation import run_open_data_chsh_validation
from xtheta.data.bell_chsh import compute_chsh_variants
from xtheta.data.schema import BellEventSchema

## 1. Manual Raw Audit

We inspect the raw file structure directly to ensure the adapter is reading the correct format.

In [ ]:
data_path = project_root / "data" / "open_bell" / "hensen" / "raw" / "bell_open_data.txt"

if not data_path.exists():
    print(f"[ERROR] Raw data not found at {data_path}")
    print("Please run: python scripts/download_open_data.py --dataset hensen")
else:
    print(f"Raw file path: {data_path.resolve()}")
    raw_lines = data_path.read_text(encoding='utf-8').splitlines()
    print(f"\nFirst 10 raw lines:")
    for line in raw_lines[:10]:
        print(line)

    df_raw = pd.read_csv(data_path, header=None)
    print(f"\nRaw total row count: {len(df_raw)}")

## 2. Apply Hensen Adapter

The adapter applies official filtering (Hensen et al., 2015) and mapping logic.

In [ ]:
if data_path.exists():
    data_iterator = load_hensen_dataset(str(data_path))
    df = pd.concat(list(data_iterator), ignore_index=True)
    print(f"Valid Bell trial count: {len(df)}")
    display(df.head())

## 3. Data Distribution Audit

Verify the distribution of settings and outcomes.

In [ ]:
if data_path.exists():
    schema = BellEventSchema()
    
    print("Unique Alice Settings:", df[schema.alice_setting].unique())
    print("Unique Bob Settings:", df[schema.bob_setting].unique())
    
    print("\nAlice Outcome Counts:")
    print(df[schema.alice_outcome].value_counts())
    
    print("\nBob Outcome Counts:")
    print(df[schema.bob_outcome].value_counts())
    
    print("\nSetting-pair counts:")
    counts = df.groupby([schema.alice_setting, schema.bob_setting]).size().reset_index(name='count')
    print(counts)

## 4. Correlation and CHSH Audit

Calculate expectations $E(a,b)$ and CHSH variants manually before running the pipeline validation.

In [ ]:
if data_path.exists():
    df['ab'] = df[schema.alice_outcome] * df[schema.bob_outcome]
    
    # Calculate E(a,b)
    expectations = df.groupby([schema.alice_setting, schema.bob_setting])['ab'].mean()
    
    E00 = expectations.get((0, 0), 0.0)
    E01 = expectations.get((0, 1), 0.0)
    E10 = expectations.get((1, 0), 0.0)
    E11 = expectations.get((1, 1), 0.0)
    
    print(f"E00: {E00:.4f}")
    print(f"E01: {E01:.4f}")
    print(f"E10: {E10:.4f}")
    print(f"E11: {E11:.4f}")

    variants = compute_chsh_variants(E00, E01, E10, E11)
    
    print("\nCHSH Sign Variants:")
    for k, v in variants.items():
        if k not in ['max_abs', 'max_abs_convention']:
            print(f"  {k}: {v:.4f}")
            
    print(f"\nMax Absolute CHSH: {variants['max_abs']:.4f} (Convention: {variants['max_abs_convention']})")

## 5. Automated Pipeline Validation

Verify the CHSH S-value and effective phase using the standardized pipeline.

In [ ]:
if data_path.exists():
    results = run_open_data_chsh_validation(
        load_hensen_dataset(str(data_path)),
        dataset_name="hensen_audit",
        output_dir="../outputs/hensen_audit",
        bootstrap_samples=1000
    )
    
    print(f"\nFinal Results:")
    print(f"S = {results['CHSH_S']:.4f} ± {results['CHSH_S_se']:.4f}")
    print(f"Phi_eff = {results['Phi_eff']:.4f}")

## 6. Conclusion

Target S (Hensen 2015): 2.42 ± 0.20.  
Observed S: {results['CHSH_S']:.4f} ± {results['CHSH_S_se']:.4f}.  
Valid Trials: {results['row_count']}.  

**Claim Classification:** This result is **open-data pipeline validation**. It confirms the framework can correctly process and interpret historical Bell-test data but does not constitute physical evidence for the X-Theta theory due to missing spacetime metadata.